# Zirconium loop line broadening derivation

This notebook derives a parameterized XRD line-broadening workflow for zirconium containing loop-driven anisotropic strain broadening.


## Section 1 — Scope and symbols

We model observed broadening as the combination of instrumental, size, and strain terms.

\[
\beta_{\mathrm{obs}}^2 \approx \beta_{\mathrm{inst}}^2 + \beta_{\mathrm{size}}^2 + \beta_{\mathrm{strain}}^2
\]

All zirconium-specific constants are external inputs and are **not** hard-coded.


In [ ]:
import numpy as np
from pathlib import Path


def _read_cif_scalars(cif_path):
    """Minimal CIF scalar reader for key-value records used in this notebook."""
    values = {}
    for raw in Path(cif_path).read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith('#') or line.startswith('loop_'):
            continue
        if not line.startswith('_'):
            continue
        parts = line.split(None, 1)
        if len(parts) != 2:
            continue
        key, value = parts
        values[key] = value.strip().strip("'").strip('"')
    return values


# Load zirconium crystallographic parameters from the bundled CIF file.
_cif_candidates = [Path('zr.cif'), Path('../zr.cif')]
for _candidate in _cif_candidates:
    if _candidate.exists():
        cif_values = _read_cif_scalars(_candidate)
        cif_path = _candidate
        break
else:
    raise FileNotFoundError('Could not locate zr.cif next to the repository/notebook.')

a_angstrom = float(cif_values['_cell_length_a'])
c_angstrom = float(cif_values['_cell_length_c'])
a_zr = a_angstrom * 1e-10
c_zr = c_angstrom * 1e-10
c_over_a_zr = c_zr / a_zr
space_group_hm = cif_values.get('_symmetry_space_group_name_H-M', 'unknown')

# Remaining broadening terms still require instrument/fit inputs.
beta_inst = 0.0018
beta_size = 0.0021
beta_strain = 0.0014
beta_obs = np.sqrt(beta_inst**2 + beta_size**2 + beta_strain**2)

print(f'Loaded CIF: {cif_path}')
print(f'Zr lattice from CIF: a={a_angstrom:.4f} Å, c={c_angstrom:.4f} Å, c/a={c_over_a_zr:.6f}')
print(f'Space group (H-M): {space_group_hm}')

beta_obs


## Section 2 — Required external inputs

The notebook now preloads zirconium crystallographic constants ($a$, $c$, $c/a$, and space group label) from `zr.cif`.
Other model parameters (contrast, defect density, instrument terms) remain explicit external inputs.

| Input | Symbol | Meaning | Units | Source |
|---|---|---|---|---|
| X-ray wavelength | $\lambda$ | Probe wavelength | m | Instrument setup |
| Zirconium lattice parameters | $a, c$ | HCP cell parameters | m | Loaded from `zr.cif` |
| Contrast baseline | $\bar{C}$ | Average dislocation contrast | – | Elastic/dislocation model |
| Contrast anisotropy coefficient | $q$ | Reflection anisotropy term | – | Elastic/dislocation model |
| Burgers vector magnitude | $b$ | Loop/dislocation Burgers magnitude | m | Defect model |
| Dislocation/loop density | $\rho$ | Defect density | m$^{-2}$ | Fitted or independent estimate |
| Shape factor | $K$ | Scherrer constant | – | Profile model assumption |
| Instrument broadening coefficients | $U,V,W$ | Caglioti parameters | rad$^2$ | Instrument calibration |


## Section 3 — Reciprocal-space transform

\[
q = \frac{4\pi}{\lambda}\sin\theta,\qquad 2\theta = 2\arcsin\left(\frac{q\lambda}{4\pi}\right)
\]


In [ ]:
from src.xrd_line_broadening import two_theta_to_q, q_to_two_theta

wavelength = 1.5406e-10

two_theta = np.deg2rad(np.array([30.0, 40.0, 50.0]))
q = two_theta_to_q(two_theta, wavelength)
two_theta_back = q_to_two_theta(q, wavelength)
q, np.rad2deg(two_theta_back)


## Section 4 — Size broadening (Scherrer)

\[
\beta_{\mathrm{size}} = \frac{K\lambda}{L\cos\theta}
\]


In [ ]:
from src.xrd_line_broadening import scherrer_fwhm

K = 0.9
L = 80e-9
theta = np.deg2rad(np.array([15.0, 20.0, 25.0]))
beta_size = scherrer_fwhm(theta, wavelength=wavelength, domain_size=L, k=K)
beta_size


## Section 5 — Strain broadening and contrast factors

Isotropic microstrain form:
\[
\beta_{\mathrm{strain}} = 4\varepsilon\tan\theta
\]

Reflection anisotropy form:
\[
C_{hkl}=\bar{C}\left(1-q\,\eta_{hkl}\right),\qquad
\eta_{hkl}=\frac{\frac{4}{3}(h^2+hk+k^2)}{\frac{4}{3}(h^2+hk+k^2)+(lc/a)^2}
\]


In [ ]:
from src.xrd_line_broadening import microstrain_fwhm
from src.contrast_factors import HKL, contrast_factor_hex

epsilon = 8e-4
beta_strain = microstrain_fwhm(theta, microstrain_rms=epsilon)

# Use c/a loaded from zr.cif; keep contrast terms as user-provided external inputs.
c_over_a = c_over_a_zr
c_bar = 0.28      # external input
q_aniso = 1.6     # external input
hkl = HKL(1, 0, 1)
C_hkl = contrast_factor_hex(hkl.h, hkl.k, hkl.l, c_over_a=c_over_a, c_bar=c_bar, q=q_aniso)

beta_strain, C_hkl


## Section 6 — Ratio framework for mixed a/c loop broadening

Starting from thesis-style strain-broadening scaling for each loop family $i\in\{a,c\}$:

\[
\beta_i^2(hkl) \propto b_i^2\,\rho_i\,C^{(i)}_{hkl}\,\Phi_i,
\]

where $b_i$ is Burgers-vector magnitude, $\rho_i$ is effective dislocation density,
$C^{(i)}_{hkl}$ is reflection-specific contrast, and $\Phi_i$ collects arrangement/correlation
terms (e.g., Wilkens-like arrangement parameter effects).

Define three ratios explicitly:

1. **Ratio of broadening contributions**
   \[
   R_\beta(hkl) \equiv \frac{\beta_a^2(hkl)}{\beta_c^2(hkl)}
   = \frac{b_a^2\rho_a C^{(a)}_{hkl}\Phi_a}{b_c^2\rho_c C^{(c)}_{hkl}\Phi_c}.
   \]

2. **Ratio of effective dislocation densities**
   \[
   R_\rho \equiv \frac{\rho_a}{\rho_c}.
   \]

3. **Ratio of loop populations**
   \[
   R_N \equiv \frac{N_a}{N_c},
   \]
   with mapping to density ratio requiring additional geometric assumptions.

Symbolic superposition for mixed a/c contributions:

\[
\beta_{\mathrm{strain,mix}}^2(hkl)=\beta_a^2(hkl)+\beta_c^2(hkl)
=K\left[b_a^2\rho_a C^{(a)}_{hkl}\Phi_a+b_c^2\rho_c C^{(c)}_{hkl}\Phi_c\right],
\]

or in ratio form using $R_\rho$:

\[
\beta_{\mathrm{strain,mix}}^2(hkl)=K\rho_c\left[b_a^2R_\rho C^{(a)}_{hkl}\Phi_a+b_c^2C^{(c)}_{hkl}\Phi_c\right].
\]

### Conditions for equivalence of ratios

To map one ratio into another, the following assumptions are required:

- **$R_\beta \leftrightarrow R_\rho$** requires known (or equal) $b_a,b_c$; known
  reflection-wise $C^{(a)}_{hkl},C^{(c)}_{hkl}$; and known/equal arrangement terms
  $\Phi_a,\Phi_c$ for the compared state/reflection.
- **$R_\rho \leftrightarrow R_N$** requires a constitutive link between loop count and
  effective density (habit-plane fractions, loop-size distributions, line length per loop,
  and visibility/effectiveness factors).
- **$R_\beta \leftrightarrow R_N$** requires both assumption sets above simultaneously.

### Missing zirconium-specific inputs for unique numeric $a$ estimate

| Missing zirconium-specific input | Role in model | Required for unique numeric $a$ estimate |
|---|---|---|
| Loop habit statistics (family fractions on basal/prismatic/pyramidal variants) | Converts population ratios to effective density terms | **Required** |
| Reflection-specific contrast factors per loop family, $C^{(a)}_{hkl},C^{(c)}_{hkl}$ | Couples anisotropy to each measured reflection | **Required** |
| Arrangement/correlation parameters per family ($\Phi_a,\Phi_c$ or equivalent Wilkens terms) | Controls screening/range effects in strain broadening | **Required** |
| Burgers-vector magnitudes and character fractions for each active family | Sets prefactor $b_i^2$ and mixed-character weighting | **Required** |
| Texture/ODF coupling assumptions for reflection weighting | Maps single-crystal contrast to polycrystal measured intensity/broadening | **Required** |
| Loop size distribution statistics by family | Needed to convert loop counts into line length/density and check consistency with size broadening | **Required** |
| Instrument-function deconvolution uncertainty bounds | Needed to isolate strain term without bias in inferred $a$ | **Required** |

> Downstream note: the Caglioti instrumental model is retained in the next code cell for
> execution of later composite-width examples.


In [ ]:
from src.xrd_line_broadening import caglioti_fwhm

U, V, W = 1.0e-5, 2.0e-5, 4.0e-5
beta_inst = caglioti_fwhm(theta, U, V, W)
beta_inst


## Section 7 — Composite width equations

For Gaussian-like components:
\[
\beta_{\mathrm{obs}} = \sqrt{\beta_{\mathrm{inst}}^2 + \beta_{\mathrm{size}}^2 + \beta_{\mathrm{strain}}^2}
\]

For Lorentzian-like components:
\[
\beta_{\mathrm{obs}} = \beta_{\mathrm{inst}} + \beta_{\mathrm{size}} + \beta_{\mathrm{strain}}
\]


In [ ]:
from src.xrd_line_broadening import combine_fwhm_gaussian, combine_fwhm_lorentzian

beta_obs_g = combine_fwhm_gaussian(beta_inst, beta_size, beta_strain)
beta_obs_l = combine_fwhm_lorentzian(beta_inst, beta_size, beta_strain)
beta_obs_g, beta_obs_l


## Section 8 — Two-population a/c loop contribution model

### Simulation label: **Parameterized sensitivity study**

The numerical example below uses illustrative scaling factors (e.g., 1.10, 0.90, 1.25) and
assumed weights. Therefore it is a **parameterized sensitivity study**, not a fully
thesis-supported prediction.

For completeness, the mixed-width form remains:

\[
\beta_{\mathrm{mix}} = \frac{w_a}{w_a+w_c}\beta_a + \frac{w_c}{w_a+w_c}\beta_c
\]

A simulation may be labeled **fully thesis-supported** only when all required zirconium
inputs listed in Section 6 are supplied from validated sources.


In [ ]:
from src.diffraction_simulation import two_population_ac_broadening

# Parameterized sensitivity study (illustrative factors; not uniquely identified).
size_a = beta_size * 1.10
strain_a = beta_strain * 0.90
size_c = beta_size * 0.95
strain_c = beta_strain * 1.25

beta_mix = two_population_ac_broadening(
    theta_rad=theta,
    instrument_fwhm_rad=beta_inst,
    size_a=size_a,
    strain_a=strain_a,
    size_c=size_c,
    strain_c=strain_c,
    weight_a=0.65,
    weight_c=0.35,
)

beta_mix


## Section 9 — Synthetic profile construction and execution checklist

Peak profile (pseudo-Voigt):
\[
I(2\theta)=\eta I_L(2\theta)+(1-\eta)I_G(2\theta)
\]

Practical execution order:
1. Enter all required external zirconium inputs.
2. Compute reflection-wise $C_{hkl}$.
3. Build size and strain widths for each reflection.
4. Combine with instrumental broadening.
5. Simulate/fitted synthetic peaks for interpretation.


In [ ]:
from src.diffraction_simulation import build_population_peaks, synthetic_pattern

x = np.deg2rad(np.linspace(25, 65, 2000))
centers = np.deg2rad(np.array([30.0, 36.0, 48.0, 57.5]))
amplitudes = np.array([1200.0, 900.0, 1400.0, 850.0])
widths = np.interp(centers, np.linspace(theta.min(), theta.max(), len(beta_mix)), beta_mix)
peaks = build_population_peaks(centers, amplitudes, widths)
pattern = synthetic_pattern(x, peaks, background=45.0)

pattern[:5], pattern.max()
